# 목표

뉴스의 카테고리 예측

In [167]:
# %load_ext colablinter

In [168]:
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gc
import os


# 시각화 관련 설정
try:
    plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
except:
    try:
        plt.rcParams['font.family'] = 'NanumGothic'
    except:
        plt.rcParams['font.family'] = 'AppleGothic'

plt.rcParams['axes.unicode_minus'] = False
fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)


# # 로그
# import logging

# def init_logger() -> logging.Logger:
#     logging.basicConfig(
#         format="%(asctime)s [%(levelname)s] (%(filename)s:%(lineno)d) - %(message)s",
#         datefmt="%Y-%m-%d %H:%M:%S",
#         level=logging.INFO,
#         encoding="utf-8",
#     )
#     return logging.getLogger("")

# logger = init_logger()

In [169]:
ROOT_DIR = os.getcwd()
DATA_DIR = os.path.join(ROOT_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "20news-bydate-train")
TEST_DIR = os.path.join(DATA_DIR, "20news-bydate-test")

In [170]:
# 파일 개수 세기
from glob import glob

train_category_list = os.listdir(TRAIN_DIR)
test_category_list = os.listdir(TEST_DIR)

train_paths = [path_ for path_ in glob(os.path.join(TRAIN_DIR, "**", "**")) 
               if os.path.isfile(path_)]
test_paths = [path_ for path_ in glob(os.path.join(TEST_DIR, "**", "**")) 
              if os.path.isfile(path_)]

print(f"· 학습 데이터: {len(train_paths)}개")
print(f"· 테스트 데이터: {len(test_paths)}개")

· 학습 데이터: 11314개
· 테스트 데이터: 7532개


학습 데이터에 비해 테스트 데이터가 너무 많은 듯?

In [171]:
train_category_set = set(train_category_list)
test_category_set = set(test_category_list)

only_in_train = train_category_set - test_category_set
only_in_test = test_category_set - train_category_set

print(f"· 학습 데이터에만 있는 카테고리: {len(only_in_train)}개")
print(f"· 테스트 데이터에만 있는 카테고리: {len(only_in_test)}개")

· 학습 데이터에만 있는 카테고리: 0개
· 테스트 데이터에만 있는 카테고리: 0개


In [172]:
CATEGORY_LIST = os.listdir(TRAIN_DIR)

In [173]:
print("[클래스별 데이터 개수]")
i = 0

for category in CATEGORY_LIST:
    i += 1

    path_list = [path_ for path_ in glob(os.path.join(TRAIN_DIR, category, "*")) 
                 if os.path.isfile(path_)]
    
    print(f"{i}) {category}: {len(path_list)}개")

[클래스별 데이터 개수]
1) talk.politics.mideast: 564개
2) rec.autos: 594개
3) comp.sys.mac.hardware: 578개
4) alt.atheism: 480개
5) rec.sport.baseball: 597개
6) comp.os.ms-windows.misc: 591개
7) rec.sport.hockey: 600개
8) sci.crypt: 595개
9) sci.med: 594개
10) talk.politics.misc: 465개
11) rec.motorcycles: 598개
12) comp.windows.x: 593개
13) comp.graphics: 584개
14) comp.sys.ibm.pc.hardware: 590개
15) sci.electronics: 591개
16) talk.politics.guns: 546개
17) sci.space: 593개
18) soc.religion.christian: 599개
19) misc.forsale: 585개
20) talk.religion.misc: 377개


- talk.religion.misc 클래스가 좀 적네

- 상위 하위 클래스로 구성되어 있으니, 클래스도 더 나눠보자.

    - misc는 잡동사니라는 뜻으로, 주요 카테고리가 아니라는 뜻이다.

    - 끝의 카테고리를 main이라고 하나 만들어서 misc와 분리하자.

- religion은 talk와 society에 겹친다.

In [174]:
TRAIN_DICT = dict()

for category in CATEGORY_LIST:
    for path_ in glob(os.path.join(TRAIN_DIR, category, "*")):

        if os.path.isfile(path_):
            
            TRAIN_DICT[path_] = dict()
            TRAIN_DICT[path_]["category"] = category

            splited_category = category.split(".")
            if splited_category[-1] != "misc":
                splited_category.append("main")

            for i in range(0, len(splited_category)):
                TRAIN_DICT[path_][i] = splited_category[i]

In [175]:
for _, dictionary in TRAIN_DICT.items():
    for i in range(1, len(dictionary) - 1):
        if dictionary[i] == "main":
            dictionary[5] = "main"
            del dictionary[i]
            continue

        elif dictionary[i] == "misc":
            dictionary[5] = "misc"
            del dictionary[i]

In [176]:
TEST_DICT = dict()

for category in CATEGORY_LIST:
    for path_ in glob(os.path.join(TEST_DIR, category, "*")):

        if os.path.isfile(path_):
            
            TEST_DICT[path_] = dict()
            TEST_DICT[path_]["category"] = category

            splited_category = category.split(".")
            if splited_category[-1] != "misc":
                splited_category.append("main")

            for i in range(0, len(splited_category)):
                TEST_DICT[path_][i] = splited_category[i]


for _, dictionary in TEST_DICT.items():
    for i in range(1, len(dictionary) - 1):
        if dictionary[i] == "main":
            dictionary[5] = "main"
            del dictionary[i]
            continue

        elif dictionary[i] == "misc":
            dictionary[5] = "misc"
            del dictionary[i]

In [177]:
# import json

# train_json_path = os.path.join(DATA_DIR, "train.json")

# with open(train_json_path, "w", encoding="utf-8") as f:
#     json.dump(TRAIN_DICT, f, indent=4)


# test_json_path = os.path.join(DATA_DIR, "test.json")

# with open(test_json_path, "w", encoding="utf-8") as f:
#     json.dump(TEST_DICT, f, indent=4)

In [178]:
count_by_class_dict = {
              0: {},
              1: {},
              2: {},
              3: {},
              4: {},
              5: {}
              }


for category_dict in TRAIN_DICT.values():
    for key, value in category_dict.items():
        if key != "category":
            if value not in count_by_class_dict[key].keys():
                count_by_class_dict[key][value] = 1
            else:
                count_by_class_dict[key][value] += 1

In [179]:
import plotly.graph_objects as go

# 데이터
data = count_by_class_dict

nodes_by_level_and_name = {}
nodes = {}
edges = set()
node_id = 0

root_key = ('ROOT', 0)
nodes[root_key] = {'id': node_id, 'label': 'ROOT', 'level': 0, 'count': len(TRAIN_DICT)}
nodes_by_level_and_name[(0, 'ROOT')] = root_key
node_id += 1

for level, name_count_dict in data.items():
    current_level = level + 1
    for name, count in name_count_dict.items():
        node_key = (name, current_level)
        if node_key not in nodes:
            nodes[node_key] = {
                'id': node_id,
                'label': name,
                'level': current_level,
                'count': count,
            }
            nodes_by_level_and_name[(current_level, name)] = node_key
            node_id += 1

for _, category_dict in TRAIN_DICT.items():
    ordered_path = sorted(
        (int(k), v) for k, v in category_dict.items() if k != 'category'
    )
    parent_key = root_key
    for level_idx, name in ordered_path:
        current_level = level_idx + 1
        node_key = nodes_by_level_and_name.get((current_level, name))
        if node_key is None:
            continue
        edges.add((parent_key, node_key))
        parent_key = node_key

levels = {}
for node_key, node in nodes.items():
    levels.setdefault(node['level'], []).append(node_key)

for level, node_keys in levels.items():
    num_nodes = len(node_keys)
    y = -level * 2
    for i, key in enumerate(node_keys):
        if num_nodes == 1:
            x = 0
        else:
            spacing = min(15, 50 / num_nodes)
            x = (i - (num_nodes - 1) / 2) * spacing
        nodes[key]['x'] = x
        nodes[key]['y'] = y

edge_x = []
edge_y = []
for parent, child in edges:
    edge_x.extend([nodes[parent]['x'], nodes[child]['x'], None])
    edge_y.extend([nodes[parent]['y'], nodes[child]['y'], None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=1.5, color='#cccccc'),
    hoverinfo='none'
)

node_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
node_x = []
node_y = []
node_labels = []
node_counts = []
node_color = []
node_sizes = []

for n in nodes.values():
    node_x.append(n['x'])
    node_y.append(n['y'])
    node_labels.append(n['label'])
    node_counts.append(str(n.get('count', 0)))
    node_color.append(node_colors[n['level'] % len(node_colors)])
    base_size = 14 if n['level'] == 0 else 10
    scale = 0 if n.get('count') is None else max(0, n['count']) ** 0.5
    node_sizes.append(base_size + scale)

parent_counts = {}
for parent, child in edges:
    parent_counts[child] = parent_counts.get(child, 0) + 1

hover_text = []
for node_key, n in nodes.items():
    parent_count = parent_counts.get(node_key, 0)
    base_text = f"{n['label']}<br>레벨: {n['level']}<br>개수: {n.get('count', 0)}"
    if parent_count > 1:
        hover_text.append(base_text + f"<br>부모 노드: {parent_count}개")
    else:
        hover_text.append(base_text)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='markers',
    marker=dict(
        size=node_sizes,
        color=node_color,
        line=dict(width=2, color='white')
    ),
    hoverinfo='text',
    hovertext=hover_text
)

count_text_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode='text',
    text=node_counts,
    textposition='middle center',
    textfont=dict(size=10, color='white', family='Arial'),
    hoverinfo='none'
)

label_text_trace = go.Scatter(
    x=node_x,
    y=[y + 0.5 for y in node_y],
    mode='text',
    text=node_labels,
    textposition='top center',
    textfont=dict(size=10, color='black', family='Arial'),
    hoverinfo='none'
)

layout = go.Layout(
    title=dict(text='카테고리 계층 구조 (중복 노드 병합)', font=dict(size=20)),
    showlegend=False,
    hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='#ffffff',
    paper_bgcolor='#ffffff',
    margin=dict(l=40, r=40, t=60, b=40),
    height=900,
)

fig = go.Figure(
    data=[edge_trace, node_trace, count_text_trace, label_text_trace],
    layout=layout,
)
fig.show()

In [ ]:
# unique한 클래스들 골라내기

unique_dict = dict()
gyep_dict = dict()
class_list = list()
class_set = set()

for category in CATEGORY_LIST:

    splited_category_list = category.split(".")[1:]

    for sub_category in splited_category_list:
        class_set.add(sub_category)


for category in CATEGORY_LIST:

    splited_category_set = set(category.split(".")[1:])

    if len()


{'atheism',
 'autos',
 'baseball',
 'christian',
 'crypt',
 'electronics',
 'forsale',
 'graphics',
 'guns',
 'hardware',
 'hockey',
 'ibm',
 'mac',
 'med',
 'mideast',
 'misc',
 'motorcycles',
 'ms-windows',
 'os',
 'pc',
 'politics',
 'religion',
 'space',
 'sport',
 'sys',
 'windows',
 'x'}

In [ ]:
import random

def get_text(path):
    with open(path, "r", encoding="utf-8") as f:
        data = f.read()
        
    return data

rand_idx = random.randint(1, len(TRAIN_DICT.keys()))
rand_key = list(TRAIN_DICT.keys())[rand_idx]

sample_key = rand_key.split("/")[-2]
sample_text = get_text(rand_key)

print(f"[샘플 출력 (label: {sample_key})]")
print("=" * 45)
print("[본문]")
print(sample_text)

[샘플 출력 (label: comp.os.ms-windows.misc)]
[본문]
From: dudek@acsu.buffalo.edu (The Cybard)
Subject: MIDI files on MS-Win3.1 and SoundBlaster 1.0?
Summary: How can I play midi files in MS-Windows 3.1 with a SB 1.0 card?
Keywords: MIDI, soundblaster, windows, ibm-pc
Organization: UB
Lines: 15
Nntp-Posting-Host: autarch.acsu.buffalo.edu

I have a 486DX-33 computer with a SoundBlaster 1.0 card.  I'm running
Microsoft Windows v3.1.  I have the SB driver set up properly to play
normal sounds (.WAV files, etc.).  I want to play midi files through the
Media Player that is included with windows.  I know I have to set up the
patch maps or something in the MIDI-Mapper in the Control Panel.  I KNOW
NOTHING ABOUT MIDI.  (This is to be the way I'll get my feet wet.)

How do I set up Windows so that I can play MIDI files?

  
-- 
David Thomas Dudek /  v098pwxs@ubvms.bitnet     \     __   _ The Cybard
 State University / dudek@sun.acsu.buffalo.edu   \   /  `-' )      ,,, 
   of New York   / "If music be 

1. '>' 같은 기호들 (단, 부등호는 조심할 것)

2. 연락처 같은 것들로 카테고리를 외운다면, 정답은 잘 맞추겠지만 과적합일 가능성이 높아질 것 같다.

3. 이메일도 지우자.
